In [ ]:
from google.colab import files
uploaded = files.upload()

Saving nakhla_cut.mp4 to nakhla_cut.mp4


In [ ]:
!pip install mediapipe opencv-python pandas


In [ ]:
!pip install --upgrade --force-reinstall numpy==1.23.5 mediapipe opencv-python


  Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.3 kB)
  Using cached mediapipe-0.10.21-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (9.7 kB)
  Using cached opencv_python-4.11.0.86-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached absl_py-2.2.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached attrs-25.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached flatbuffers-25.2.10-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached jax-0.6.1-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.6.1-cp311-cp311-manylinux2014_x86_64.whl.metadata (1.2 kB)
  Using cached matplotlib-3.10.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached opencv_contrib_python-4.11.0.86-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached protobuf-4.25.7-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
  Using cached sounddevice-0.5.

In [ ]:
#extract pictures
import cv2
import mediapipe as mp
import os


# Initialize MediaPipe
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils


hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.3,
    min_tracking_confidence=0.3
)


# Load video
video_path = 'nakhla_cut.mp4'  # Make sure this file exists
cap = cv2.VideoCapture(video_path)


# Create output directory
output_folder = "nakhla_landmarks"
os.makedirs(output_folder, exist_ok=True)


frame_idx = 0
print("Starting extraction...")


while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break


    # Convert BGR to RGB
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(image_rgb)


    # Draw landmarks if detected
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)


    # Save annotated frame
    output_path = os.path.join(output_folder, f"frame_{frame_idx:04d}.jpg")
    cv2.imwrite(output_path, frame)


    frame_idx += 1


cap.release()
print(f"Extraction complete. Saved {frame_idx} frames to '{output_folder}'")



Starting extraction...
Extraction complete. Saved 12299 frames to 'nakhla_landmarks'


In [ ]:
import cv2
import mediapipe as mp
import os
import csv


frames_folder = "nakhla_landmarks"
output_csv = "nakhla_landmarks.csv"


mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=2,
    min_detection_confidence=0.3
)


skipped_frames = 0
no_hand_detected = 0
landmark_written = 0


with open(output_csv, mode='w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['frame_name', 'hand_index', 'landmark_id', 'x', 'y', 'z'])

    frame_files = sorted(os.listdir(frames_folder))
    print(f"Processing {len(frame_files)} frames for landmark extraction...")


    for frame_name in frame_files:
        frame_path = os.path.join(frames_folder, frame_name)
        image = cv2.imread(frame_path)


        if image is None or image.sum() == 0:
            skipped_frames += 1
            continue


        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)


        if not results.multi_hand_landmarks:
            no_hand_detected += 1
            continue


        for hand_idx, hand_landmarks in enumerate(results.multi_hand_landmarks):
            for landmark_id, landmark in enumerate(hand_landmarks.landmark):
                csv_writer.writerow([
                    frame_name,
                    hand_idx,
                    landmark_id,
                    landmark.x,
                    landmark.y,
                    landmark.z
                ])
                landmark_written += 1


print(f"✅ Extraction complete.")
print(f"🟢 Landmarks written: {landmark_written}")
print(f"⚠️ Skipped unreadable frames: {skipped_frames}")
print(f"⚠️ Skipped (no hand detected): {no_hand_detected}")


Processing 12299 frames for landmark extraction...
✅ Extraction complete.
🟢 Landmarks written: 44562
⚠️ Skipped unreadable frames: 0
⚠️ Skipped (no hand detected): 10189
